# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading and exploring the FAIR^2 clinical oncology dataset using the `mlcroissant` library.

## Dataset Source
The dataset is described via a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the dataset object
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata information
print(f"Dataset name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Let's examine the available record sets, their fields, and the unique `@id` values for reference.

If you are unsure which record sets are present, you can list all record sets and their fields using the schema.

In [ ]:
# List available record sets and their fields by @id
print("Available record sets and fields:")
record_set_ids = []
for rs in dataset.record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print("  Fields:")
    for field in rs.get('field', []):
        # field may be a dict or @id string
        if isinstance(field, dict):
            print(f"    - {field['@id']}")
        else:
            print(f"    - {field}")

## 3. Data Extraction
Load records from the available record sets into pandas DataFrames.

_Update the `record_sets_of_interest` list to include all record set `@id`s you want to analyze._

In [ ]:
# Select all record set @id's for extraction (see previous cell's output)
record_sets_of_interest = record_set_ids.copy()  # Use all record sets by default

# Load records for each record set into DataFrames
dataframes = {}
for rs_id in record_sets_of_interest:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set {rs_id}")

# Display columns of the main (first) record set
if record_sets_of_interest:
    main_rs_id = record_sets_of_interest[0]
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze numerical variables from the main record set.

- We'll identify a numeric field and a grouping field for demonstration.
- Filter records, compute a normalized score, and perform groupby statistics by a key attribute.

_Change the field `@id`s below based on your schema's output above as needed._

In [ ]:
# Assign the record set @id and field @ids by inspecting previous outputs
record_set_id = main_rs_id
df = dataframes[record_set_id]

# Attempt to detect numeric and grouping fields
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
group_field_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # Use first detected numeric column
    print(f"Numeric field selected: {numeric_field_id}")
else:
    raise ValueError("No numeric field found.")

# Select a group field (fallback to the second column if candidates empty)
group_field_id = group_field_candidates[0] if group_field_candidates else df.columns[1]
print(f"Grouping by field: {group_field_id}")

# Set threshold for demonstration (10th percentile)
threshold = df[numeric_field_id].quantile(0.1)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records in {record_set_id} with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Add normalized column
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nFirst 5 records with normalized '{numeric_field_id}':")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Plot distribution and groupwise statistics for selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=20, color='navy')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group
if group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to:

- Load FAIR^2 clinical dataset metadata and record sets via a Croissant schema URL
- Examine available record sets and their fields using `@id`
- Extract data from the record set(s) into pandas DataFrames
- Perform exploratory analysis, including numeric normalization and groupby statistics
- Visualize variable distributions and grouped differences

This approach can be adapted to other Croissant-compliant datasets for transparent and reproducible analysis workflows.